# DAVID-Net Training â€” Kaggle

In [ ]:
# Cell 1: GPU check + install deps
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU.")
!pip install -q transformers accelerate scikit-learn jiwer datasets
import shutil; assert shutil.which('ffmpeg'), 'ffmpeg missing - decoding needs it'

In [ ]:
# Cell 2: Clone repo
import os, sys
REPO = "/kaggle/working/david-net-av"
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/MIHMahmudEli/david-net-av.git {REPO}
sys.path.insert(0, REPO)
print(f"Repo: {REPO}")

In [ ]:
# Cell 3: Load HF token
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
print("HF token loaded.")

In [ ]:
# Cell 4: Discover + Auto-download all 8 datasets
import subprocess
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
DOWNLOAD_DIR = WORKING / "kaggle_datasets"
DOWNLOAD_DIR.mkdir(exist_ok=True)

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}
AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg"}

def has_media_files(d):
    for f in d.rglob("*"):
        if f.suffix.lower() in VIDEO_EXTS | AUDIO_EXTS:
            return True
    return False

DATASETS = {
    "fakeavceleb": {
        "slug": "aicontentdetections/fakeavceleb-v1-2",
        "mounts": ["aicontentdetections/fakeavceleb-v1-2", "aicontentdetections/FakeAVCeleb_v1.2"],
    },
    "dfdc-10": {
        "slug": "pranay22077/dfdc-10",
        "mounts": ["pranay22077/dfdc-10"],
    },
    "deepfaketimit": {
        "slug": "fahimaislam1812/deepfaketimit",
        "mounts": ["fahimaislam1812/deepfaketimit", "fahimaislam1812/DeepfakeTIMIT"],
    },
    "celeb-df-v2": {
        "slug": "reubensuju/celeb-df-v2",
        "mounts": ["reubensuju/celeb-df-v2"],
    },
    "asvpoof-2019": {
        "slug": "anishsarkar22/asvpoof-2019-dataset-la",
        "mounts": ["anishsarkar22/asvpoof-2019-dataset-la"],
    },
    "in-the-wild": {
        "slug": "abdallamohamed312/in-the-wild-audio-deepfake",
        "mounts": ["abdallamohamed312/in-the-wild-audio-deepfake"],
    },
    "wavefake": {
        "slug": "walimuhammadahmad/fakeaudio",
        "mounts": ["walimuhammadahmad/fakeaudio", "andreadiubaldo/wavefake-test"],
    },
}

def find_mounted(slug_paths):
    for p in slug_paths:
        for base in [KAGGLE_INPUT, KAGGLE_INPUT / "datasets"]:
            candidate = base / p
            if candidate.exists():
                return candidate
    return None

def download_dataset(slug, friendly):
    dst = DOWNLOAD_DIR / friendly
    if dst.exists() and has_media_files(dst):
        print(f"  {friendly}: already downloaded")
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    print(f"  {friendly}: downloading from {slug}...", end=" ")
    try:
        result = subprocess.run(
            ["kaggle", "datasets", "download", "-d", slug, "-p", str(dst), "--unzip"],
            capture_output=True, text=True, timeout=3600
        )
        if result.returncode == 0:
            print("OK")
            return dst
        else:
            print(f"FAIL: {result.stderr[:200]}")
            return None
    except Exception as e:
        print(f"ERROR: {e}")
        return None

datasets = {}
for friendly, info in DATASETS.items():
    mounted = find_mounted(info["mounts"])
    if mounted:
        datasets[friendly] = mounted
        print(f"  {friendly} -> {mounted} (mounted)")
        continue
    downloaded = DOWNLOAD_DIR / friendly
    if downloaded.exists() and has_media_files(downloaded):
        datasets[friendly] = downloaded
        print(f"  {friendly} -> {downloaded} (cached)")
        continue
    result = download_dataset(info["slug"], friendly)
    if result:
        datasets[friendly] = result

print(f"\nFound {len(datasets)}/8 datasets.")
if len(datasets) < 8:
    missing = set(DATASETS) - set(datasets)
    print(f"Missing: {missing}")

In [ ]:
# Cell 5: Extract compressed datasets (multi-part zips, tar, etc.)
import zipfile, tarfile, shutil

WORKING = Path("/kaggle/working")
DATA_DIR = WORKING / "data"
DATA_DIR.mkdir(exist_ok=True)

def find_first_zip_part(src):
    """Find the .001 part of a multi-part zip, anywhere in tree."""
    best = None
    best_num = 999999
    count = 0
    for f in src.rglob("*"):
        name = f.name
        # Match patterns like: foo.zip.001, foo.zip.016
        if ".zip." in name:
            parts = name.split(".zip.")
            if len(parts) == 2 and parts[1].isdigit():
                count += 1
                num = int(parts[1])
                if num < best_num:
                    best_num = num
                    best = f
    if best:
        print(f"    Found {count} zip parts, first = {best.name} (part {best_num})")
    return best, count

def extract_multipart_zip(name, first_part, dst, search_root=None):
    print(f"  {name}: extracting multi-part zip from {first_part.name}...")
    if search_root is None:
        search_root = first_part.parent
    stem = first_part.name.split(".zip.")[0]
    # Find ALL parts across all subdirs
    all_parts = sorted(search_root.rglob(f"{stem}.zip.*"),
                       key=lambda x: int(x.name.split(".zip.")[1]))
    print(f"  {name}: found {len(all_parts)} parts across subdirs")

    # Method 1: Try 7z â€” copy all parts to temp dir first (7z needs them co-located)
    try:
        import shutil, tempfile
        with tempfile.TemporaryDirectory() as tmpdir:
            tmp = Path(tmpdir)
            for p in all_parts:
                shutil.copy2(str(p), str(tmp / p.name))
            first_tmp = tmp / first_part.name
            result = subprocess.run(
                ["7z", "x", str(first_tmp), f"-o{dst}", "-y"],
                capture_output=True, text=True, timeout=600
            )
            if result.returncode == 0:
                print(f"  {name}: OK (via 7z)")
                return True
            print(f"  {name}: 7z failed: {result.stderr[:200]}")
    except FileNotFoundError:
        print(f"  {name}: 7z not found, trying concat method...")
    except subprocess.TimeoutExpired:
        print(f"  {name}: 7z timed out")

    # Method 2: Concatenate all parts into one zip, then extract
    # Search from dataset root (not just .001 parent) since parts may be in sibling dirs
    # Method 2: Concatenate all parts into one zip, then extract
    try:
        print(f"  {name}: concatenating {len(all_parts)} parts...")
        merged = dst / f"{stem}_merged.zip"
        with open(merged, "wb") as out:
            for part in all_parts:
                with open(part, "rb") as inp:
                    while True:
                        chunk = inp.read(8 * 1024 * 1024)
                        if not chunk:
                            break
                        out.write(chunk)
        print(f"  {name}: merged to {merged.stat().st_size // 1024 // 1024}MB, extracting...")
        with zipfile.ZipFile(str(merged)) as zf:
            zf.extractall(dst)
        merged.unlink()  # remove merged zip to save space
        print(f"  {name}: OK (via concat)")
        return True
    except Exception as e:
        print(f"  {name}: FAILED: {e}")
        return False

def extract_archives(name, src, dst):
    extracted = False
    # Multi-part zip first â€” pass src as search_root so we find parts across all subdirs
    first_part, count = find_first_zip_part(src)
    if first_part:
        extracted = extract_multipart_zip(name, first_part, dst, search_root=src)
    # Regular zips
    for arch in src.rglob("*.zip"):
        if ".zip." in arch.name:
            continue
        print(f"  {name}: extracting {arch.name}...", end=" ")
        try:
            with zipfile.ZipFile(arch) as zf:
                zf.extractall(dst)
            print("OK")
            extracted = True
        except Exception as e:
            print(f"FAIL: {e}")
    # Tar.gz
    for arch in src.rglob("*.tar.gz"):
        print(f"  {name}: extracting {arch.name}...", end=" ")
        try:
            with tarfile.open(arch, "r:gz") as tf:
                tf.extractall(dst)
            print("OK")
            extracted = True
        except Exception as e:
            print(f"FAIL: {e}")
    return extracted

for name, path in datasets.items():
    dst = DATA_DIR / name
    dst.mkdir(exist_ok=True)
    marker = dst / ".extracted"
    if marker.exists():
        print(f"  {name}: already extracted")
        continue
    if has_media_files(path):
        print(f"  {name}: loose media files found")
        marker.touch()
        continue
    # Check disk space â€” skip if dataset too large for Kaggle (~20GB working)
    import shutil as _shutil
    free_gb = _shutil.disk_usage(str(DATA_DIR)).free / (1024**3)
    zip_count = sum(1 for _ in path.rglob("*.zip.*") if '.zip.' in _.name and _.name.split('.zip.')[1].isdigit())
    est_gb = zip_count * 1.0  # each part is ~1GB
    if est_gb > free_gb * 0.85:
        print(f"  {name}: SKIPPED â€” {est_gb:.0f}GB needed but only {free_gb:.1f}GB free")
        continue
    print(f"  {name}: no loose media, searching archives...")
    try:
        extract_archives(name, path, dst)
    except OSError as e:
        print(f"  {name}: FAILED (disk error): {e}")
        # Clean up partial extraction to free space
        import shutil as _shutil2
        if dst.exists():
            _shutil2.rmtree(dst, ignore_errors=True)
        continue
    if has_media_files(dst):
        print(f"  {name}: extracted media OK")
    else:
        print(f"  {name}: WARNING - no media files after extraction")
    marker.touch()

print("\nExtraction done.")

In [ ]:
# Cell 6: Build ALL manifests
MANIFEST_DIR = WORKING / "manifests"
MANIFEST_DIR.mkdir(exist_ok=True)
SPLIT_DIR = WORKING / "splits"
SPLIT_DIR.mkdir(exist_ok=True)

# === FakeAVCeleb ===
fakeav_root = datasets.get("fakeavceleb")
if fakeav_root:
    for candidate in [fakeav_root, fakeav_root / "FakeAVCeleb_v1.2"]:
        if any((candidate / q).exists() for q in ["RealVideo-RealAudio", "FakeVideo-FakeAudio"]):
            fakeav_root = candidate
            break
    # The manifest's rel_path is relative to THIS directory -> training must use the
    # same root. (Run 1 passed the mount root instead; every clip was "missing" and the
    # loader silently trained on random tensors. The loader now raises instead.)
    datasets["fakeavceleb"] = fakeav_root
    print(f"FakeAVCeleb: {fakeav_root}")
    !cd {REPO} && python scripts/build_manifest.py \
        --root {fakeav_root} \
        --out {MANIFEST_DIR}/fakeavceleb.jsonl \
        --splits-dir {SPLIT_DIR}/fakeavceleb --seed 42

# === All other datasets ===
CONVERTERS = [
    ("dfdc-10", "dfdc-10"),
    ("deepfaketimit", "deepfaketimit"),
    ("celeb-df-v2", "celeb-df-v2"),
    ("asvpoof-2019", "asvpoof-2019"),
    ("in-the-wild", "in-the-wild"),
    ("wavefake", "wavefake"),
]

for ds_name, dataset_key in CONVERTERS:
    root = datasets.get(dataset_key)
    extracted_root = DATA_DIR / ds_name
    if not root or not root.exists():
        if extracted_root.exists() and has_media_files(extracted_root):
            root = extracted_root
            print(f"  {ds_name}: using extracted path {root}")
    if root and root.exists():
        print(f"\n--- {ds_name} ---")
        !cd {REPO} && python scripts/build_manifests.py \
            --dataset {dataset_key} \
            --root {root} \
            --out {MANIFEST_DIR}/{ds_name}.jsonl \
            --splits-dir {SPLIT_DIR}/{ds_name}
    else:
        print(f"  {ds_name}: NOT FOUND")

print("\n" + "="*50)
print("ALL MANIFESTS:")
for f in sorted(MANIFEST_DIR.glob("*.jsonl")):
    !wc -l {f}

In [ ]:
# Cell 7: Training manifests = SUBJECT-DISJOINT SPLITS (never the full manifest)
import json, yaml
from collections import Counter

FAKEAV_ROOT = str(datasets["fakeavceleb"])          # resolved in Cell 6 (contains RealVideo-RealAudio/ ...)
FAKEAV_SPLITS = SPLIT_DIR / "fakeavceleb"
TRAIN_MANIFEST = FAKEAV_SPLITS / "train.jsonl"
VAL_MANIFEST = FAKEAV_SPLITS / "val.jsonl"
TEST_MANIFEST = FAKEAV_SPLITS / "test.jsonl"
for m in (TRAIN_MANIFEST, VAL_MANIFEST, TEST_MANIFEST):
    assert m.exists(), f"missing split {m} - re-run Cell 6"

def _summary(path):
    recs = [json.loads(l) for l in open(path) if l.strip()]
    return len(recs), dict(Counter(r["quadrant"] for r in recs)), dict(Counter(r["generator"] for r in recs))

for name, m in [("train", TRAIN_MANIFEST), ("val", VAL_MANIFEST), ("test", TEST_MANIFEST)]:
    n, quads, gens = _summary(m)
    print(f"{name:5s}: {n:6d} clips  quadrants={quads}")
    if name == "train":
        print(f"       generators={gens}")

# Hard preflight: the first train clip must exist under FAKEAV_ROOT and decode to real
# frames/audio (this is the check that would have caught run 1 at t=0).
first = json.loads(open(TRAIN_MANIFEST).readline())
probe = Path(FAKEAV_ROOT) / first["rel_path"]
assert probe.exists(), f"{probe} does not exist -> FAKEAV_ROOT is wrong"
from src.data.decode import decode_clip
d = decode_clip(str(probe), 16, 64000, 224, window="center")
print(f"preflight OK: {probe.name} video{tuple(d.video.shape)} std={d.video.std():.3f} "
      f"audio{tuple(d.audio.shape)} std={d.audio.std():.4f} has_audio={d.has_audio}")
assert d.has_audio and d.audio.std() > 1e-4 and d.video.std() > 1e-3, "decoded clip looks empty"


In [ ]:
# Cell 8: QACP Stage 0 config (pristine RVRA clips of the TRAIN split only)
QACP_CONFIG = {
    "run_id": "qacp_stage0_v2",          # new run_id: the v1 checkpoint was trained on random tensors
    "d_model": 768, "n_heads": 8, "n_fusion_layers": 4, "dropout": 0.1,
    "use_sync": True, "use_disentangle": True, "compose_quadrant": False,
    "video_backbone": "videomae", "audio_backbone": "wavlm",
    "video_model_name": "MCG-NJU/videomae-base", "audio_model_name": "microsoft/wavlm-base-plus",
    "freeze_blocks": 8, "freeze_feature_extractor": True,
    "init_from": None,
    "n_frames": 16, "audio_len": 64000, "shard_root": None, "feature_cache": None,
    "train_manifest": str(TRAIN_MANIFEST),          # filtered to RVRA inside the script
    "root_dir": FAKEAV_ROOT,
    "qacp_views_per_clip": 4,                        # ~350 reals x 4 pseudo-views per epoch
    "batch_size": 4, "grad_accum_steps": 8, "num_workers": 4, "epochs": 15,
    "milestone_every": 5, "keep_milestones": 2,
    "lr": 1e-4, "lr_encoder": 1e-5, "weight_decay": 1e-4, "max_grad_norm": 1.0,
    "warmup_epochs": 1, "gradient_checkpointing": True,
    "patience": 5, "min_delta": 1e-3,
    "log_every": 10,
    "out_dir": str(WORKING / "runs"), "local_dir": str(WORKING),
    "qacp_temperature": 0.2, "seed": 42,
}

qacp_path = WORKING / "qacp_config.yaml"
with open(qacp_path, "w") as f:
    yaml.dump(QACP_CONFIG, f)
print(f"QACP config -> {qacp_path}")


In [ ]:
# Cell 9: Run QACP Stage 0
!cd {REPO} && python -m src.training.pretrain_qacp \
    --config {qacp_path} \
    --run-id {QACP_CONFIG['run_id']}

In [ ]:
# Cell 10: Stage 1 config (init from QACP best checkpoint on HF) -- 3 seeds
import glob
from pathlib import Path

# -- Download the QACP best checkpoint from HF --------------------------------
from src.utils.hf_backup import HFBackup

QACP_RUN_ID = QACP_CONFIG["run_id"]   # "qacp_stage0"
_qacp_dl_dir = WORKING / "qacp_ckpt"
_qacp_dl_dir.mkdir(parents=True, exist_ok=True)

qacp_backup = HFBackup(run_id=QACP_RUN_ID, local_dir=str(WORKING))
print(f"Downloading QACP best checkpoint from HF (run_id={QACP_RUN_ID})...")

qacp_ckpt = qacp_backup.download_best(local_dir=str(_qacp_dl_dir))
if qacp_ckpt is None:
    print("  best.pt not found -- trying latest checkpoint...")
    qacp_ckpt = qacp_backup.download_latest(local_dir=str(_qacp_dl_dir))
if qacp_ckpt is None:
    print("WARNING: No QACP checkpoint found on HF.\n"
          "Stage 1 will train from RANDOM INIT (sub-optimal but safe).\n"
          "Re-run Cell 9 first to generate a QACP checkpoint.")
else:
    print(f"QACP checkpoint: {qacp_ckpt}")

# -- Build Stage-1 configs (3 seeds) ------------------------------------------
SEEDS = [42, 123, 456]
STAGE1_CONFIGS = []

for seed in SEEDS:
    cfg = {
        "run_id": f"stage1_v2_seed{seed}",
        "d_model": 768, "n_heads": 8, "n_fusion_layers": 4, "dropout": 0.1,
        "use_sync": True, "use_disentangle": True, "compose_quadrant": False,
        "video_backbone": "videomae", "audio_backbone": "wavlm",
        "video_model_name": "MCG-NJU/videomae-base",
        "audio_model_name": "microsoft/wavlm-base-plus",
        "freeze_blocks": 6, "freeze_feature_extractor": True,
        # Use downloaded QACP weights; None falls back to random init gracefully
        "init_from": qacp_ckpt if (qacp_ckpt and Path(qacp_ckpt).exists()) else None,
        "n_frames": 16, "audio_len": 64000, "shard_root": None,
        "feature_cache": None,
        "train_manifest": str(TRAIN_MANIFEST),       # train split ONLY (val/test are held out)
        "val_manifest": str(VAL_MANIFEST),
        "root_dir": FAKEAV_ROOT,
        "modality_dropout": 0.15, "augment": True,
        # 15k train clips / 4 = ~3.8k micro-steps per epoch on a T4 (~30-40 min);
        # 10 epochs x 3 seeds spans several 12 h sessions -> HF resume handles it.
        "batch_size": 4, "grad_accum_steps": 4, "num_workers": 4, "epochs": 10,
        "milestone_every": 5, "keep_milestones": 2,
        "lr": 1e-4, "lr_encoder": 1e-5, "weight_decay": 1e-4, "max_grad_norm": 1.0,
        "warmup_epochs": 1, "gradient_checkpointing": True,
        "nan_patience": 5, "max_nan_restores": 3,
        "log_every": 25,
        "out_dir": str(WORKING / "runs"), "local_dir": str(WORKING),
        "loss_weights": {"v": 1.0, "a": 1.0, "quad": 0.5, "loc": 0.5,
                         "sync": 0.1, "disentangle": 0.1},
        "seed": seed,
    }
    path = WORKING / f"stage1_seed{seed}_config.yaml"
    with open(path, "w") as f:
        yaml.dump(cfg, f)
    STAGE1_CONFIGS.append((seed, path, cfg))
    init_tag = "from QACP" if cfg["init_from"] else "random init"
    print(f"Seed {seed}: {cfg['run_id']}  ({init_tag})")


In [ ]:
# Cell 11: Run Stage 1 training (3 seeds)
for seed, config_path, cfg in STAGE1_CONFIGS:
    print(f"\n{'='*60}")
    print(f"Training seed {seed}...")
    print(f"{'='*60}")
    !cd {REPO} && python -m src.training.train \
        --config {config_path} \
        --run-id {cfg['run_id']}

In [ ]:
# Cell 12: In-domain test + cross-dataset evaluation (with HF backup)
import json

# In-domain FakeAVCeleb test split FIRST (Table 3.1), then cross-dataset (Table 3.2).
EVAL_DATASETS = {
    "fakeavceleb-test": (TEST_MANIFEST, FAKEAV_ROOT, 0),
    "celeb-df-v2":      (MANIFEST_DIR / "celeb-df-v2.jsonl", datasets.get("celeb-df-v2"), 0),
    "dfdc-10":          (MANIFEST_DIR / "dfdc-10.jsonl", datasets.get("dfdc-10"), 4000),
    "deepfaketimit":    (MANIFEST_DIR / "deepfaketimit.jsonl", datasets.get("deepfaketimit"), 0),
    "in-the-wild":      (MANIFEST_DIR / "in-the-wild.jsonl", datasets.get("in-the-wild"), 6000),
    "asvpoof-2019":     (MANIFEST_DIR / "asvpoof-2019.jsonl", datasets.get("asvpoof-2019"), 6000),
    "wavefake":         (MANIFEST_DIR / "wavefake.jsonl", datasets.get("wavefake"), 4000),
}
# max_clips > 0 = deterministic stratified subsample (WaveFake alone is 134k files)

all_results = {}
for seed, config_path, stage1_cfg in STAGE1_CONFIGS:
    run_id = stage1_cfg["run_id"]
    best_ckpt = WORKING / "runs" / run_id / "best.pt"          # written by train.py
    if not best_ckpt.exists():
        print(f"Seed {seed}: no local best.pt, downloading from HF...")
        hf = HFBackup(run_id=run_id, local_dir=str(WORKING))
        best_ckpt = hf.download_best(local_dir=str(WORKING / "dl" / run_id))
        if not best_ckpt:
            print(f"  No checkpoint on HF - skipping seed {seed}")
            continue

    seed_results = {}
    for ds_name, (manifest, root, max_clips) in EVAL_DATASETS.items():
        if manifest is None or not Path(manifest).exists() or root is None:
            print(f"  {ds_name}: no manifest/root, skipping")
            continue
        print(f"\nSeed {seed} on {ds_name}...")
        report_path = WORKING / f"eval_{run_id}_{ds_name}.json"
        !cd {REPO} && python -m src.eval.evaluate \
            --config {config_path} \
            --checkpoint {best_ckpt} \
            --manifest {manifest} \
            --root-dir {root} \
            --max-clips {max_clips} \
            --out {report_path} \
            --run-id {run_id} \
            --ds-name {ds_name} \
            --skip-if-done
        if report_path.exists():
            with open(report_path) as f:
                r = json.load(f)
            seed_results[ds_name] = {
                "video_auc": r["video"]["auc"],
                "audio_auc": r["audio"]["auc"],
                "quadrant_acc": r["quadrant"]["acc"],
                "n": r["n"],
            }
            print(f"  v_auc={r['video']['auc']} a_auc={r['audio']['auc']} quad_acc={r['quadrant']['acc']:.4f} n={r['n']}")
    all_results[f"seed{seed}"] = seed_results

print("\n" + "="*70)
print("EVALUATION SUMMARY  (AUC is NaN when a manifest has a single class for that modality)")
print("="*70)
print(f"{'Dataset':<20} {'Type':<8} {'V-AUC':<10} {'A-AUC':<10} {'Quad-Acc':<10} {'n':<8}")
print("-"*70)
for seed_key, ds_results in all_results.items():
    for ds, m in ds_results.items():
        dtype = "AV" if ds in ["fakeavceleb-test", "dfdc-10", "deepfaketimit", "celeb-df-v2"] else "Audio"
        va = "nan" if m['video_auc'] != m['video_auc'] else f"{m['video_auc']:.4f}"
        aa = "nan" if m['audio_auc'] != m['audio_auc'] else f"{m['audio_auc']:.4f}"
        print(f"{ds:<20} {dtype:<8} {va:<10} {aa:<10} {m['quadrant_acc']:<10.4f} {m['n']:<8}")


In [ ]:
# Cell 13: Robustness sweep (Fig. A, RQ5) + ALL manuscript artifacts -> HF paper/
# Uses the first seed with a checkpoint. Robustness = blur / downscale / quantize / SNR sweeps
# on a stratified 600-clip subset of the in-domain test split (tensor-space degradations).
best_seed_cfg = None
for seed, config_path, cfg in STAGE1_CONFIGS:
    ck = WORKING / "runs" / cfg["run_id"] / "best.pt"
    if not ck.exists():
        ck = Path(str(WORKING / "dl" / cfg["run_id"] / "runs" / cfg["run_id"] / "best" / "best.pt"))
    if ck.exists():
        best_seed_cfg = (seed, config_path, cfg, ck)
        break
assert best_seed_cfg, "no trained checkpoint found - run Cell 11 first"
seed, config_path, cfg, best_ckpt = best_seed_cfg
ROBUSTNESS_JSON = WORKING / f"robustness_{cfg['run_id']}.json"
if not ROBUSTNESS_JSON.exists():
    !cd {REPO} && python -m src.eval.robustness         --config {config_path} --checkpoint {best_ckpt}         --manifest {TEST_MANIFEST} --root-dir {FAKEAV_ROOT} --max-clips 600         --out {ROBUSTNESS_JSON}

RUN_IDS = " ".join(c["run_id"] for _, _, c in STAGE1_CONFIGS)
!cd {REPO} && python scripts/paper_artifacts.py     --work {WORKING} --out {WORKING}/paper     --run-ids {RUN_IDS} --qacp-run-id {QACP_CONFIG['run_id']}     --in-domain fakeavceleb-test     --robustness {ROBUSTNESS_JSON} --config {config_path} --checkpoint {best_ckpt}     --push
# -> HF: paper/metrics/{summary.json, per_generator.json, efficiency.json, tables.tex, eval/*},
#        paper/figures/results_{roc, roc_cross_dataset, reliability, confusion, localization,
#        robustness, training_curves, per_generator}.{pdf,png}, paper/logs/*, paper/configs/*


In [ ]:
# Cell 14: Publish the model for END USERS -> public HF model repo (weights + config + model card + API)
# The backup repo is private and full of optimizer states; users get MoshinAli/david-net-av.
PUBLIC_MODEL_REPO = "MoshinAli/david-net-av"
!cd {REPO} && python scripts/publish_model.py     --checkpoint {best_ckpt} --config {config_path}     --summary {WORKING}/paper/metrics/summary.json     --repo {PUBLIC_MODEL_REPO} --public --repo-root {REPO}
print(f"Model card: https://huggingface.co/{PUBLIC_MODEL_REPO}")


In [ ]:
# Cell 15: Final summary (from paper/metrics/summary.json)
import json
S = json.load(open(WORKING / "paper" / "metrics" / "summary.json"))
print("="*78)
print("DAVID-Net — results (mean +/- std over seeds", S["run_ids"], ")")
print("="*78)
print(f"{'Dataset':<22}{'n':>7}{'V-AUC':>16}{'A-AUC':>16}{'Quad-Acc':>16}")
for ds, d in S["datasets"].items():
    a = d["aggregate"]; n = next(iter(d["per_seed"].values()))["n"]
    f = lambda k: "--" if a[k]["mean"] != a[k]["mean"] else f"{a[k]['mean']:.4f}+/-{a[k]['std']:.4f}"
    tag = "*" if ds == S["in_domain"] else " "
    print(f"{ds+tag:<22}{n:>7}{f('video_auc'):>16}{f('audio_auc'):>16}{f('quadrant_acc'):>16}")
print("* = in-domain test split; -- = single-class corpus for that stream")
print("
On HF (MoshinAli/david-net-av-backup):")
print("  runs/<run_id>/{checkpoints,best,logs,eval,state}   per-run training state")
print("  paper/metrics/   summary.json, per_generator.json, efficiency.json, tables.tex, eval/")
print("  paper/figures/   ROC (in-domain + cross-dataset), reliability, confusion, localization,")
print("                   robustness, training curves, per-generator")
print("  paper/logs/, paper/configs/")
print("Public model for end users: https://huggingface.co/MoshinAli/david-net-av")
